<a href="https://colab.research.google.com/github/scativa/IV-Colab/blob/eval250127test/Eval_segmentation_pretrained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Librerias


In [2]:
# Reemplaza la línea `import functions as func`

import numpy as np
import os
import cv2
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import fnmatch

def load_images(dir, filenames=None, norm=False, file_pattern=None, resize_to=(224, 224)):
  file_pattern = "*" if file_pattern is None else file_pattern
  filenames = [f for f in os.listdir(dir) if fnmatch.fnmatch(f, file_pattern)] if filenames is None else filenames
  print(f"Cargando {len(filenames)} imagenes... ")

  images = []
  filenames_read = []
  for filename in filenames:
    image = cv2.imread(os.path.join(dir, filename))
    if image is not None:
      images.append(image.astype(np.float32))
      filenames_read.append(filename)
  if norm:
      images = [image / 255.0 for image in images]

  # https://chatgpt.com/c/67fd5724-8164-8006-9367-780189493ee5
  images = [cv2.resize(image, resize_to).astype(np.float32) for image in images]

  return images, filenames_read

# def load_images(dir, norm=False):
    # images = [cv2.imread(os.path.join(dir, filename)).astype(np.float32)
    #         for filename in os.listdir(dir)
    #         if cv2.imread(os.path.join(dir, filename)) is not None]
    # if norm:
    #     images = [image / 255.0 for image in images]

    # # https://chatgpt.com/c/67fd5724-8164-8006-9367-780189493ee5
    # images = [cv2.resize(image, (224, 224)).astype(np.float32) for image in images]
    # return images

def load_masks(dir_pores, image_filenames):
    masks = []
    for filename in image_filenames:
        mask_path = os.path.join(dir_pores, filename)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)

        # Normalizar
        mask = mask / 255.0

        # Redimensionar
        mask = cv2.resize(mask, (224, 224))

        # Binarizar DESPUÉS del resize
        mask = (mask > 0.5).astype(np.float32)

        masks.append(mask)

    return masks
'''def load_masks(dir_pores, image_filenames):
    masks = [
            cv2.imread(os.path.join(dir_pores, filename), cv2.IMREAD_GRAYSCALE).astype(np.float32)
            # if os.path.exists(os.path.join(dir_pores, filename))
            # else np.zeros((232, 180), dtype=np.float32)
            for filename in image_filenames
    ]
    # masks = [np.expand_dims(mask, axis=-1) for mask in masks] #adds a channel dimension to the masks

    # https://chatgpt.com/c/67fd5724-8164-8006-9367-780189493ee5
    masks = [cv2.resize(mask, (224, 224)).astype(np.float32) for mask in masks]

    return masks'''

# just to plot masks
def plot_img_mask(img, mask, pred):

    sub_plot = 0
    num_cols = 0

    # Determina el número de columnas en función de si hay una predicción
    if img is not None:
        num_cols += 1
    if mask is not None:
        num_cols += 1
    if pred is not None:
        num_cols += 1

    # Si no hay nada para mostrar, sal de la función
    if num_cols == 0:
        print("No hay nada para mostrar.")
        return

    # Crea la figura y los ejes en una sola fila con el número determinado de columnas
    fig, axes = plt.subplots(1, num_cols, figsize=(3, 3))

    if (img is not None):
      axes[sub_plot].imshow(img)
      axes[sub_plot].set_title('Image')
      axes[sub_plot].axis('off')
      sub_plot += 1

    if (mask is not None):
      axes[sub_plot].imshow(mask, cmap='gray')
      axes[sub_plot].set_title('Mask')
      axes[sub_plot].axis('off')
      sub_plot += 1

    if (pred is not None):
      pred_mask = np.where(pred <= 0.5, 0, 1)

      axes[sub_plot].imshow(pred_mask, cmap='gray')
      axes[sub_plot].set_title('Predicted')
      axes[sub_plot].axis('off')

    plt.tight_layout()
    plt.show()

def augmented_dataset(dir_images, dir_mask, seed, norm):
    # From "Pores 03/28/25 -Seba- Basado en segmentation_own (DTE_fisuras).ipynb"


    images = load_images(dir_images, norm)
    # Get image filenames from loaded images

    image_filenames = [os.path.basename(image_path) for image_path in os.listdir(dir_images)]
    # masks = load_masks(dir_pores)
    masks = load_masks(dir_mask, image_filenames)


    X_train, X_test, y_train, y_test = train_test_split(images, masks, test_size=0.2, random_state=seed)

    X_train[0].shape
    y_train[0].shape
    augmented_X_train = [cv2.flip(i, 1) for i in X_train]
    augmented_y_train = [cv2.flip(i, 1) for i in y_train]

    X_train = np.array(X_train + augmented_X_train)
    y_train = np.array(y_train + augmented_y_train)

    X_test = np.array(X_test)
    y_test = np.array(y_test)

    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)

    y_train = np.expand_dims(y_train, axis=-1)
    y_val = np.expand_dims(y_val, axis=-1)
    y_test = np.expand_dims(y_test, axis=-1)

    return X_train, y_train, X_val, y_val, X_test, y_test

def test_dataset(dir_images, dir_mask, seed, norm):
    # From "Pores 03/28/25 -Seba- Basado en segmentation_own (DTE_fisuras).ipynb"


    images = load_images(dir_images, norm)
    # Get image filenames from loaded images

    image_filenames = [os.path.basename(image_path) for image_path in os.listdir(dir_images)]
    # masks = load_masks(dir_pores)
    masks = load_masks(dir_mask, image_filenames)


    _, X_test, _, y_test = train_test_split(images, masks, test_size=0.2, random_state=seed)

    y_test = np.expand_dims(y_test, axis=-1)

    return X_test, y_test

'''
If you want to show some other metrics, just uncomment
'''
def plot_history(history, x_lim, y_lim):
    #val_iou_score = history.history['val_iou_score']
    #iou_score = history.history['iou_score']
    val_f1_score = history.history['val_f1-score']
    #f1_score = history.history['f1-score']
    val_loss = history.history['val_loss']
    loss = history.history['loss']
    epochs = range(1, len(loss) + 1)

    # Plotting
    plt.figure(figsize=(12, 6))

    #plt.plot(epochs, val_iou_score, 'b', label='Validation IoU Score')
    #plt.plot(epochs, iou_score, 'g', label='IoU Score')
    #plt.plot(epochs, f1_score, 'r', label='F1 Score')
    plt.plot(epochs, val_f1_score, 'c', label='Validation F1 Score')
    plt.plot(epochs, val_loss, 'm', label='Validation Loss')
    plt.plot(epochs, loss, 'y', label='Training Loss')

    plt.title('Training Metrics', fontsize=16)
    plt.xlabel('Epochs', fontsize=14)
    plt.ylabel('Score/Loss', fontsize=14)
    plt.legend(fontsize=12)

    plt.grid(True)
    plt.xlim(1, x_lim)
    plt.ylim(0, y_lim)
    plt.show()



##Librerías Test
---
Ver de unificar con las librerías generales

In [3]:
import matplotlib.pyplot as plt
import os
import numpy as np

# ===
# === Carga y procesamiento de datos ====
# ===
def load_X(image_directory, save_dir, fname, modo, file_pattern=None, mmap_mode='r'):
  # Usar 'cargar' para usar los datos ya procesados guardados (.npy)
  # Usar 'generar' para crear nuevos datos y guardarlos a partir de las imágenes originales
  if modo == 'cargar':
      if not file_pattern is None:
        print("Warning: file_pattern is ignored in 'cargar' mode.")
      print("Cargando datos preprocesados desde archivos .npy...")
      X  = np.load(os.path.join(save_dir, f'{fname}.npy'), mmap_mode=mmap_mode)
      # X_gen  = np.load(os.path.join(save_dir, f'X_gen{npy_suf}.npy'), mmap_mode='r')
      with open(os.path.join(save_dir, f'{fname}.txt'), "r") as f:
      # with open(os.path.join(save_dir, f'X_gen{npy_suf}.txt'), "r") as f:
        filenames = [line.strip() for line in f]
      # https://chatgpt.com/share/68927496-27fc-8010-a17b-f906d9f5e1c9
      print("✅ Datos cargados correctamente.")

  elif modo == 'generar':
      print("Generando nuevos datos a partir de imágenes... {}")
      images, filenames = load_images(image_directory, norm=False, file_pattern=file_pattern)
      X = np.array(images)
      print(X.shape)

      # Guardar los datos generados
      np.save(os.path.join(save_dir, f'{fname}.npy'), X)
      with open(os.path.join(save_dir, f'{fname}.txt'), "w") as f:
        f.writelines(name + "\n" for name in filenames)

      print(f"✅ Nuevos datos generados y guardados en {save_dir} {fname}.npy")
  else:
      raise ValueError("❌ Modo inválido. Usa 'cargar' o 'generar'.")

  return X, filenames

def preprocesar_X(X, save_dir=None, out_fn=None, mmap_mode='r'):
  if (save_dir is not None) and (out_fn is not None) and os.path.exists(os.path.join(save_dir, out_fn)):
    print(f"Cargando datos preprocesados desde archivos {out_fn}... ",end="")
    # https://chatgpt.com/share/68927496-27fc-8010-a17b-f906d9f5e1c9
    print(os.path.join(save_dir, out_fn), os.path.exists(os.path.join(save_dir, out_fn)))
    X_preprocessed  = np.load(os.path.join(save_dir, out_fn), mmap_mode=mmap_mode)
    if X_preprocessed is not None:
      print(f"✅ Datos PREPROCESADOS cargados correctamente de {out_fn}.")
    else:
      print(f"❌ No se encontraron datos preprocesados en {out_fn}.")
  else:
    print("Preprocesando datos... ",end="")

    preprocessed_input = get_preprocessing(model_type)
    # Preprocesar las imágenes
    X_preprocessed = preprocessed_input(np.copy(X))
    if (save_dir is not None) and (out_fn is not None):
      np.save(os.path.join(save_dir, out_fn), X_preprocessed)
      print(f"✅ Datos preprocesados y guardados correctamente en {os.path.join(save_dir, out_fn)}.")

  return X_preprocessed

In [4]:
# ===
# === Carga de modelo ====
# ===
# https://chatgpt.com/share/68925d8f-359c-8010-8d19-5297230ba987
# Es preciso para cargar el modelo ejecutar el código de definición de los custom objects
def load_model(model_path):
  # === 5. Cargar o crear modelo ===
  # Ensure the directory exists before attempting to load the model
  # os.makedirs(os.path.dirname(final_model_path), exist_ok=True)
  print(f"Attempting to load model from: {final_model_path}") # Added print statement
  if os.path.exists(final_model_path):
      print(f"🔁 Cargando modelo completo desde: {final_model_path}")
      model = tf.keras.models.load_model(
          final_model_path,
          custom_objects={
              'custom_bce_dice_loss': custom_bce_dice_loss,
              'custom_iou_score': custom_iou_score,
              'custom_f1_score': custom_f1_score
          }
      )
      if model:
        print("Carga exitosa")
        return model
      else:
        print(f"Carga fallida {final_model_path}")
        return None
  else:
    print(f"Modelo inexistente {final_model_path}")
    return None


In [5]:

# ===
# === Guardar resultados ====
# ===
def save_results(y_pred, filenames, output_dir, solo_mask=False):
  # Define the base directory for output and predictions
  # output_dir = f"{base_path}/output_XN_avg"
  predictions_dir = os.path.join(output_dir, f"predictions_{defect}_{model_type}")
  results_dir = os.path.join(predictions_dir, f"visualizations_{defect}_{model_type}") # Keep visualization directory as before
  pred_dir = os.path.join(predictions_dir, f"pred_masks_{defect}_{model_type}") # Keep visualization directory as before

  # Create the necessary directories if they don't exist
  os.makedirs(predictions_dir, exist_ok=True)
  if not solo_mask:
    os.makedirs(results_dir, exist_ok=True)
  os.makedirs(pred_dir, exist_ok=True)

  # # === 2. Guardar predicciones como .npy ===
  # np.save(os.path.join(predictions_dir, "y_pred_gen.npy"), y_pred)
  # print("✅ Predicciones guardadas.")

  # === 3. Visualizar y guardar comparaciones ===
  # Escala las máscaras para visualización (umbral opcional)
  threshold = 0.5
  y_pred_bin = (y_pred > threshold).astype(np.uint8)

  # Iterar sobre algunas muestras (puedes limitar si es mucho)
  for i in range(len(y_pred)):
      if not solo_mask:
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        original_image_display = np.clip(X_gen[i], 0, 255).astype(np.uint8)

        # Imagen original
        axes[0].imshow(original_image_display)
        axes[0].set_title("Imagen original")
        axes[0].axis("off")

        # # Ground truth
        # axes[1].imshow(y_test[i].squeeze(), cmap='gray')
        # axes[1].set_title("Máscara Real")
        # axes[1].axis("off")

        # Predicción
        axes[2].imshow(y_pred_bin[i].squeeze(), cmap='gray')
        axes[2].set_title("Predicción")
        axes[2].axis("off")

        plt.tight_layout()
        save_path = os.path.join(results_dir, f"test_{filenames[i]}")
        plt.savefig(save_path)
        plt.close()

      pred_path = os.path.join(pred_dir, f"pred_{filenames[i]}")
      cv2.imwrite(pred_path, (y_pred_bin[i].squeeze()) * 255)
      # plt.imsave(mask_path, y_pred, cmap='gray')

  print(f"🖼️ Visualizaciones guardadas en: {results_dir}")


##Informe


---
###Test segmentación
Notebook dedicada a probar con distintas imágenes las redes generadas en entrenamientos de VGG19, Resnet152 y Efficientnet

Con base en las notebook de Evilus y modificadas para evaluarlas con el muestreo [250516](https://github.com/scativa/IV-Colab/tree/eval250516):

* Efficientnet_Cachaduras_07_28_25_segmentation_pretrained.ipynb
* Efficientnet_Pores__072825_segmentation_pretrained.ipynb
* Resnet152_Cachaduras_07_28_25_segmentation_pretrained.ipynb
* Resnet152_Pores__072825_segmentation_pretrained.ipynb
* VGG_Pores_072825_segmentation_pretrained.ipynb
* Vgg_Cachaduras_07_28_25_segmentation_pretrainedx.ipynb
* efficientnet_Pores__072825_segmentation_pretrained.ipynb

#Entorno

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from time import process_time_ns
import os, json

usr_drive = "evi.cnea"

json_path = os.path.join('/content/drive/MyDrive/', 'drvinfo.json')
usr_drive = json.load(open(json_path)).get('username') if os.path.exists(json_path) else json_path

base_paths = { # Path al
    "ppca.cnea": '/content/drive/MyDrive/IV',
    "seba.cnea": '/content/drive/MyDrive/CNEA/DCA/Proyectos/IV - Inspección Visual/IV',
    "scativa": '/content/drive/MyDrive/Laburo/CNEA/DCA/Proyectos/IV - Inspección Visual/IV',
    "evi.cnea": '/content/drive/MyDrive/IV',
    "evi55": '/content/drive/MyDrive/IV'
}
base_path = base_paths[usr_drive]

if not os.path.exists(base_path):
  print(f'Carpeta inexistente "{usr_drive}: {base_path}"')
else:
  print(f'Utilizando carpeta  "{usr_drive}: {base_path}"')

os.environ["SM_FRAMEWORK"] = "tf.keras"
from tensorflow import keras
from keras import backend as K


Utilizando carpeta  "scativa: /content/drive/MyDrive/Laburo/CNEA/DCA/Proyectos/IV - Inspección Visual/IV"


In [8]:
!pip install segmentation-models
import segmentation_models as sm
from segmentation_models import get_preprocessing

Segmentation Models: using `tf.keras` framework.


In [9]:
import segmentation_models as sm
import tensorflow as tf

# === Registro para serialización ===
@tf.keras.utils.register_keras_serializable(package="Custom")
def custom_bce_dice_loss(y_true, y_pred):
    return sm.losses.bce_dice_loss(y_true, y_pred)

@tf.keras.utils.register_keras_serializable(package="Custom")
def custom_iou_score(y_true, y_pred):
    return sm.metrics.iou_score(y_true, y_pred)

@tf.keras.utils.register_keras_serializable(package="Custom")
def custom_f1_score(y_true, y_pred):
    return sm.metrics.f1_score(y_true, y_pred)


---

#Test

---



---


## Test de Generalización


---


Prueba con otras imágenes que no forma parte del dataset


###Ejecución de Test


---



In [10]:
# Parámetros y entorno
defect = "pores" # 'cachaduras'
muestreo = "20250127"
data_id = "BA_test_s42" # "XN_p" "BA_test_s42"
model_type = "vgg19"
model_id = "20250729_120611" if defect == 'pores' else "20250728_205521"

# === Rutas ===
image_directory = f'{base_path}/Images/{muestreo}/{data_id}'

if not os.path.exists(image_directory):
  print(f'Carpeta inexistente "{usr_drive}: {image_directory}"')
else:
  print(f'Utilizando carpeta  "{usr_drive}: {image_directory}"')

save_dir = f'{base_path}/processed_data/{muestreo}_{data_id}' # El preprocesamiento no depende del defecto, sí del tipo de red y de la forma de entrenamiento
os.makedirs(save_dir, exist_ok=True)

Utilizando carpeta  "scativa: /content/drive/MyDrive/Laburo/CNEA/DCA/Proyectos/IV - Inspección Visual/IV/Images/20250127/BA_test_s42"


In [11]:
# === Carga de Modelo ===
model_fn = f"{model_id}-{model_type}-{defect}"
final_model_path = f"{base_path}/072925_Output, processed data y modelos/{defect}/{model_type}/{model_fn}.keras"

model = load_model(final_model_path)

Attempting to load model from: /content/drive/MyDrive/Laburo/CNEA/DCA/Proyectos/IV - Inspección Visual/IV/072925_Output, processed data y modelos/pores/vgg19/20250729_120611-vgg19-pores.keras
🔁 Cargando modelo completo desde: /content/drive/MyDrive/Laburo/CNEA/DCA/Proyectos/IV - Inspección Visual/IV/072925_Output, processed data y modelos/pores/vgg19/20250729_120611-vgg19-pores.keras
Carga exitosa


### Generar .npy

In [ ]:
# Procesados
# for et in [26000, 28000, 30000, 32000, 34000, 36000, 38000]:
for et in [28000, 30000, 32000, 34000, 36000, 38000]:
  # === Carga de datos ===
  print(f"{et}) ")
  X_gen, filenames = load_X(image_directory, save_dir, f'X_gen_{data_id}_et{et}', 'generar', file_pattern = f"*-ET{et}.00-*.*", mmap_mode='r')

print("Fin de procesamiento")


28000) 
Generando nuevos datos a partir de imágenes... {}
Cargando 4800 imagenes... 
(4800, 224, 224, 3)
✅ Nuevos datos generados y guardados en /content/drive/MyDrive/Laburo/CNEA/DCA/Proyectos/IV - Inspección Visual/IV/processed_data/20250127_BA_test_s42 X_gen_BA_test_s42_et28000.npy
30000) 
Generando nuevos datos a partir de imágenes... {}
Cargando 4800 imagenes... 


In [ ]:
import time

# Preprocesados
for et in [28000, 30000, 32000, 34000, 36000, 38000]:
# for et in [26000, 28000, 30000, 32000, 34000, 36000, 38000]:
  # === Carga de datos ===
  print(f"{et}) ")

  # X_gen, filenames = load_X(image_directory, save_dir, f'X_gen_{data_id}_et{et}', 'generar', file_pattern = f"*-ET{et}.00-*.*", mmap_mode='r')
  X_gen, filenames = load_X(image_directory, save_dir, f'X_gen_{data_id}_et{et}', 'cargar', mmap_mode='r')

  # === Preprocesamiento ===
  print(f'Preprocesamiento para: {model_type}')
  print(time.strftime("%Y-%m-%d %H:%M:%S"))
  # X_gen_preprocessed = preprocesar_X(X_gen)
  X_gen_preprocessed = preprocesar_X(X_gen, save_dir, f'X_gen_{data_id}_et{et}_{model_type}.npy', mmap_mode='r')
  print(time.strftime("%Y-%m-%d %H:%M:%S"))

###Predicción

In [ ]:
import time

for et in [26000, 28000, 30000, 32000, 34000, 36000, 38000]:
  # === Carga de datos ===
  print(f"{et}) ")
  X_gen, filenames = load_X(image_directory, save_dir, f'X_gen_{data_id}_et{et}', 'cargar', mmap_mode='r') # Ver la opción de solo generar los filenames

  # === Preprocesamiento ===
  print(f'Preprocesamiento para: {model_type}')
  print(time.strftime("%Y-%m-%d %H:%M:%S"))
  # X_gen_preprocessed = preprocesar_X(X_gen)
  # X_gen_preprocessed = preprocesar_X(X_gen, save_dir, f'X_gen_{data_id}_et{et}_{model_type}.npy', mmap_mode='r')
  X_gen_preprocessed = preprocesar_X(None, save_dir, f'X_gen_{data_id}_et{et}_{model_type}.npy', mmap_mode='r')
  print(time.strftime("%Y-%m-%d %H:%M:%S"))
  print(len(X_gen_preprocessed))

  # === Predicción ===
  print(f'Predicción para: {model_type}')
  print(time.strftime("%Y-%m-%d %H:%M:%S"))
  y_pred = model.predict(X_gen_preprocessed)

  # === Guardar resultados ===
  print(f'Guardando resultados para: {model_type}')
  print(time.strftime("%Y-%m-%d %H:%M:%S"))
  save_results(y_pred, filenames, f"{base_path}/output_{data_id}/{et}", solo_mask=True)
